In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(r"../datasets")
PROCESSED_DIR = DATA_DIR / "processed"

train_ratings = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
test_ratings = pd.read_csv(PROCESSED_DIR / "ratings_test.csv")
movies = pd.read_csv(PROCESSED_DIR / "phase2_movies.csv")
weighted_matrix = load_npz(PROCESSED_DIR / "weighted_matrix.npz")

print("Train:", train_ratings.shape)
print("Test:", test_ratings.shape)
print("Movies:", movies.shape)
print("Weighted matrix:", weighted_matrix.shape)

assert len(movies) == weighted_matrix.shape[0]


In [12]:
movie_id_to_index = pd.Series(
    movies.index.to_numpy(),
    index=movies["movieId"]
)

assert movies["movieId"].is_unique


In [13]:
## Build a user preference vector

POSITIVE_THRESHOLD = 4.0

def build_user_vector(user_id):
    history = train_ratings[
        (train_ratings["userId"] == user_id) &
        (train_ratings["rating"] >= POSITIVE_THRESHOLD)
    ].copy()

    history["matrix_index"] = history["movieId"].map(movie_id_to_index)
    history = history.dropna(subset=["matrix_index"])

    if history.empty:
        return None

    indices = history["matrix_index"].astype(int).to_numpy()
    vectors = weighted_matrix[indices]
    weights = history["rating"].to_numpy(dtype=float) - 3.0

    return csr_matrix(
        vectors.multiply(weights[:, None]).sum(axis=0) / weights.sum()
    )


In [14]:
## Generate personalized recommendations
def recommend_for_user(user_id, n=10):
    user_vector = build_user_vector(user_id)

    if user_vector is None:
        return pd.DataFrame()

    scores = cosine_similarity(
        user_vector,
        weighted_matrix
    ).ravel()

    result = movies.copy()
    result["content_score"] = scores

    rated_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    result = result[
        ~result["movieId"].isin(rated_ids)
    ]

    return (
        result
        .sort_values("content_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


In [ ]:
test_user = int(train_ratings["userId"].iloc[0])

recommendations = recommend_for_user(test_user, 10)

display(recommendations)


In [ ]:
## Evaluate the personalized content model
positive_test = test_ratings[
    test_ratings["rating"] >= POSITIVE_THRESHOLD
]

relevant_by_user = (
    positive_test
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

def evaluate(k):
    rows = []

    for user_id, relevant in relevant_by_user.items():
        recs = recommend_for_user(user_id, k)

        if recs.empty:
            continue

        predicted = set(recs["movieId"])
        hits = len(predicted & relevant)

        rows.append({
            "userId": user_id,
            "k": k,
            "precision": hits / k,
            "recall": hits / len(relevant),
            "hit_rate": int(hits > 0)
        })

    return pd.DataFrame(rows)

results = pd.concat(
    [evaluate(5), evaluate(10)],
    ignore_index=True
)

display(
    results.groupby("k")
    .agg(
        precision_at_k=("precision","mean"),
        recall_at_k=("recall","mean"),
        hit_rate_at_k=("hit_rate","mean"),
        users=("userId","nunique")
    )
    .reset_index()
)


In [ ]:
#  Phase 4 validation
assert weighted_matrix.shape[0] == len(movies)
assert movies["movieId"].is_unique
assert not recommendations["movieId"].isin(
    train_ratings.loc[
        train_ratings["userId"] == test_user,
        "movieId"
    ]
).any()

print("PHASE 4 PERSONALIZED CONTENT VALIDATION PASSED")
